In [56]:
# Synthetic 1 m Land-Cover Map (500×500) — notebook one-cell version
# Edit ONLY the FRAC dict (and BASE_OUT if desired), then run.

from pathlib import Path
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ===================== USER CONTROLS =====================
BASE_OUT = Path(r"G:\Hangkai\Anttarctic Vegetation Dynamic\DART\Antarctic_NDVI_all\Antarctic_NDVI_all\Data\LandCover\TableA")  # where the auto-named folder will be created
SIZE_M   = 500                   # map size in meters/pixels (1 m per pixel)
KERNEL   = 25                    # odd number; larger -> smoother snow/algae blobs
SEED     = 43                    # RNG seed for reproducibility

# ---- Control all fractions here (sum can be <= 1.0; remainder is rock) ----
FRAC = {
    "snow":   0.00,   # class 0 (clustered)
    "algae":  0.00,  # class 3 (clustered)
    "moss":   0.00,  # class 2 (sparse on rock)
    "lichen": 0.49,  # class 4 (sparse on rock)
}
NORMALIZE_IF_SUM_GT_1 = True     # If True, auto-normalize when sum > 1.0. If False, raise error.
# =========================================================

# ---------- helpers ----------
def _box_blur_2d(a: np.ndarray, k: int) -> np.ndarray:
    assert k % 2 == 1, "KERNEL must be odd."
    ker = np.ones(k, dtype=np.float32) / k
    tmp = np.apply_along_axis(lambda v: np.convolve(v, ker, mode="same"), axis=1, arr=a)
    out = np.apply_along_axis(lambda v: np.convolve(v, ker, mode="same"), axis=0, arr=tmp)
    return out

def _norm01(x: np.ndarray) -> np.ndarray:
    x = x - x.min()
    xmax = x.max()
    return x / xmax if xmax > 0 else x

def _top_k_mask(field: np.ndarray, k: int, eligible: np.ndarray) -> np.ndarray:
    if k <= 0: return np.zeros_like(eligible, dtype=bool)
    idx = np.flatnonzero(eligible)
    if idx.size == 0: return np.zeros_like(eligible, dtype=bool)
    vals = field.ravel()[idx]
    if k >= idx.size:
        m = np.zeros_like(eligible, dtype=bool)
        m.flat[idx] = True
        return m
    thresh_idx = np.argpartition(vals, -k)[-k:]
    chosen = idx[thresh_idx]
    m = np.zeros_like(eligible, dtype=bool)
    m.flat[chosen] = True
    return m

# ---------- generator ----------
def generate_landcover(size_m, kernel, snow_frac, algae_frac, moss_frac, lichen_frac, seed=43):
    rng = np.random.default_rng(seed)
    N = size_m * size_m

    req = np.array([snow_frac, algae_frac, moss_frac, lichen_frac], dtype=float)
    req = np.clip(req, 0.0, 1.0)
    s = float(req.sum())
    if s > 1.0:
        if NORMALIZE_IF_SUM_GT_1:
            req = req / s
        else:
            raise ValueError(f"Fractions sum to {s:.3f} (>1). Lower them or set NORMALIZE_IF_SUM_GT_1=True.")
    snow_frac, algae_frac, moss_frac, lichen_frac = req.tolist()

    snow_n   = int(round(snow_frac   * N))
    algae_n  = int(round(algae_frac  * N))
    moss_n   = int(round(moss_frac   * N))
    lichen_n = int(round(lichen_frac * N))

    # clustered fields for snow & algae
    snow_raw  = rng.standard_normal((size_m, size_m)).astype(np.float32)
    algae_raw = rng.standard_normal((size_m, size_m)).astype(np.float32)
    snow_s  = _norm01(_box_blur_2d(snow_raw,  kernel))
    algae_s = _norm01(_box_blur_2d(algae_raw, kernel))

    land = np.full((size_m, size_m), 1, dtype=np.uint8)  # rock=1

    snow_mask  = _top_k_mask(snow_s,  snow_n,  np.ones_like(snow_s, dtype=bool))
    land[snow_mask] = 0
    algae_mask = _top_k_mask(algae_s, algae_n, ~snow_mask)
    land[algae_mask] = 3

    rock_mask = ~(snow_mask | algae_mask)

    # moss (random on rock)
    rock_idx = np.flatnonzero(rock_mask)
    if rock_idx.size > 0 and moss_n > 0:
        chosen = rng.choice(rock_idx, size=min(moss_n, rock_idx.size), replace=False)
        moss_mask = np.zeros_like(rock_mask, dtype=bool); moss_mask.flat[chosen] = True
        land[moss_mask] = 2
        rock_mask &= ~moss_mask

    # lichen (random on leftover rock)
    rock_idx2 = np.flatnonzero(rock_mask)
    if rock_idx2.size > 0 and lichen_n > 0:
        chosen2 = rng.choice(rock_idx2, size=min(lichen_n, rock_idx2.size), replace=False)
        lichen_mask = np.zeros_like(rock_mask, dtype=bool); lichen_mask.flat[chosen2] = True
        land[lichen_mask] = 4

    # return array + the (maybe adjusted) fractions
    return land, dict(snow=snow_frac, algae=algae_frac, moss=moss_frac, lichen=lichen_frac)

# ---------- save & preview ----------
PALETTE = {
    0: (230/255, 230/255, 230/255), # snow
    1: (130/255, 120/255, 110/255), # rock
    2: ( 50/255, 160/255,  80/255), # moss
    3: (100/255, 190/255, 200/255), # algae
    4: (200/255, 180/255,  60/255), # lichen
}
CLASS_NAMES = {0:"snow",1:"rock",2:"moss",3:"algae",4:"lichen"}

def _colorize(arr: np.ndarray) -> np.ndarray:
    rgb = np.zeros((arr.shape[0], arr.shape[1], 3), dtype=np.float32)
    for cls, col in PALETTE.items(): rgb[arr == cls] = col
    return rgb

def save_outputs(land: np.ndarray, base_out_dir: Path, fdict: dict):
    orig_snow   = int(round(fdict['snow']   * 100, 0))
    orig_algae  = int(round(fdict['algae']  * 100, 0))
    orig_moss   = int(round(fdict['moss']   * 100, 0))
    orig_lichen = int(round(fdict['lichen'] * 100, 0))
    tag = f"Snow_{orig_snow}_Algae_{orig_algae}_Moss_{orig_moss}_Lichen_{orig_lichen}"

    out_dir = base_out_dir / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    # array + csv
    np.save(out_dir / "lc_map.npy", land)
    pd.DataFrame(land).to_csv(out_dir / "lc_map.csv", header=False, index=False)

    # metadata
    meta = dict(size_m=int(land.shape[0]), kernel=KERNEL, seed=SEED, **fdict, classes=CLASS_NAMES)
    (out_dir / "metadata.json").write_text(json.dumps(meta, indent=2))

    # png preview with legend
    rgb = _colorize(land)
    fig, ax = plt.subplots(figsize=(6,6), dpi=200)
    ax.imshow(rgb, interpolation="nearest"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("Synthetic Land-Cover (1 m)")
    legend_handles = [Patch(color=PALETTE[c], label=f"{c} = {CLASS_NAMES[c]}") for c in sorted(PALETTE)]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=7, frameon=True)
    fig.tight_layout(); fig.savefig(out_dir / "lc_map.png", bbox_inches="tight"); plt.close(fig)
    return out_dir

# ---------- run ----------
BASE_OUT.mkdir(parents=True, exist_ok=True)
land, used_fracs = generate_landcover(
    size_m=SIZE_M, kernel=KERNEL,
    snow_frac=FRAC["snow"], algae_frac=FRAC["algae"],
    moss_frac=FRAC["moss"], lichen_frac=FRAC["lichen"],
    seed=SEED
)
out_dir = save_outputs(land, BASE_OUT, used_fracs)

# quick summary in notebook output
uniq, cnt = np.unique(land, return_counts=True)
print("Saved to:", out_dir)
print("Pixel counts:", dict(zip(uniq.tolist(), cnt.tolist())))

Saved to: G:\Hangkai\Anttarctic Vegetation Dynamic\DART\Antarctic_NDVI_all\Antarctic_NDVI_all\Data\LandCover\TableA\Snow_0_Algae_0_Moss_0_Lichen_49
Pixel counts: {1: 127500, 4: 122500}


In [258]:
# Synthetic 1 m Land-Cover Map (500×500) — notebook one-cell version
# Edit ONLY the FRAC dict (and BASE_OUT if desired), then run.

from pathlib import Path
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ===================== USER CONTROLS =====================
BASE_OUT = Path(r"G:\Hangkai\Anttarctic Vegetation Dynamic\DART\Antarctic_NDVI_all\Antarctic_NDVI_all\Data\LandCover\TableB")  # where the auto-named folder will be created
SIZE_M   = 500                   # map size in meters/pixels (1 m per pixel)
KERNEL   = 25                    # odd number; larger -> smoother snow/algae blobs
SEED     = 43                    # RNG seed for reproducibility

# ---- Control all fractions here (sum can be <= 1.0; remainder is rock) ----
FRAC = {
    "snow":   0.05,   # class 0 (clustered)
    "algae":  0.10,  # class 3 (clustered)
    "moss":   0.10,  # class 2 (sparse on rock)
    "lichen": 0.10,  # class 4 (sparse on rock)
}
NORMALIZE_IF_SUM_GT_1 = True     # If True, auto-normalize when sum > 1.0. If False, raise error.
# =========================================================

# ---------- helpers ----------
def _box_blur_2d(a: np.ndarray, k: int) -> np.ndarray:
    assert k % 2 == 1, "KERNEL must be odd."
    ker = np.ones(k, dtype=np.float32) / k
    tmp = np.apply_along_axis(lambda v: np.convolve(v, ker, mode="same"), axis=1, arr=a)
    out = np.apply_along_axis(lambda v: np.convolve(v, ker, mode="same"), axis=0, arr=tmp)
    return out

def _norm01(x: np.ndarray) -> np.ndarray:
    x = x - x.min()
    xmax = x.max()
    return x / xmax if xmax > 0 else x

def _top_k_mask(field: np.ndarray, k: int, eligible: np.ndarray) -> np.ndarray:
    if k <= 0: return np.zeros_like(eligible, dtype=bool)
    idx = np.flatnonzero(eligible)
    if idx.size == 0: return np.zeros_like(eligible, dtype=bool)
    vals = field.ravel()[idx]
    if k >= idx.size:
        m = np.zeros_like(eligible, dtype=bool)
        m.flat[idx] = True
        return m
    thresh_idx = np.argpartition(vals, -k)[-k:]
    chosen = idx[thresh_idx]
    m = np.zeros_like(eligible, dtype=bool)
    m.flat[chosen] = True
    return m

# ---------- generator ----------
def generate_landcover(size_m, kernel, snow_frac, algae_frac, moss_frac, lichen_frac, seed=43):
    rng = np.random.default_rng(seed)
    N = size_m * size_m

    req = np.array([snow_frac, algae_frac, moss_frac, lichen_frac], dtype=float)
    req = np.clip(req, 0.0, 1.0)
    s = float(req.sum())
    if s > 1.0:
        if NORMALIZE_IF_SUM_GT_1:
            req = req / s
        else:
            raise ValueError(f"Fractions sum to {s:.3f} (>1). Lower them or set NORMALIZE_IF_SUM_GT_1=True.")
    snow_frac, algae_frac, moss_frac, lichen_frac = req.tolist()

    snow_n   = int(round(snow_frac   * N))
    algae_n  = int(round(algae_frac  * N))
    moss_n   = int(round(moss_frac   * N))
    lichen_n = int(round(lichen_frac * N))

    # clustered fields for snow & algae
    snow_raw  = rng.standard_normal((size_m, size_m)).astype(np.float32)
    algae_raw = rng.standard_normal((size_m, size_m)).astype(np.float32)
    snow_s  = _norm01(_box_blur_2d(snow_raw,  kernel))
    algae_s = _norm01(_box_blur_2d(algae_raw, kernel))

    land = np.full((size_m, size_m), 1, dtype=np.uint8)  # rock=1

    snow_mask  = _top_k_mask(snow_s,  snow_n,  np.ones_like(snow_s, dtype=bool))
    land[snow_mask] = 0
    algae_mask = _top_k_mask(algae_s, algae_n, ~snow_mask)
    land[algae_mask] = 3

    rock_mask = ~(snow_mask | algae_mask)

    # moss (random on rock)
    rock_idx = np.flatnonzero(rock_mask)
    if rock_idx.size > 0 and moss_n > 0:
        chosen = rng.choice(rock_idx, size=min(moss_n, rock_idx.size), replace=False)
        moss_mask = np.zeros_like(rock_mask, dtype=bool); moss_mask.flat[chosen] = True
        land[moss_mask] = 2
        rock_mask &= ~moss_mask

    # lichen (random on leftover rock)
    rock_idx2 = np.flatnonzero(rock_mask)
    if rock_idx2.size > 0 and lichen_n > 0:
        chosen2 = rng.choice(rock_idx2, size=min(lichen_n, rock_idx2.size), replace=False)
        lichen_mask = np.zeros_like(rock_mask, dtype=bool); lichen_mask.flat[chosen2] = True
        land[lichen_mask] = 4

    # return array + the (maybe adjusted) fractions
    return land, dict(snow=snow_frac, algae=algae_frac, moss=moss_frac, lichen=lichen_frac)

# ---------- save & preview ----------
PALETTE = {
    0: (230/255, 230/255, 230/255), # snow
    1: (130/255, 120/255, 110/255), # rock
    2: ( 50/255, 160/255,  80/255), # moss
    3: (100/255, 190/255, 200/255), # algae
    4: (200/255, 180/255,  60/255), # lichen
}
CLASS_NAMES = {0:"snow",1:"rock",2:"moss",3:"algae",4:"lichen"}

def _colorize(arr: np.ndarray) -> np.ndarray:
    rgb = np.zeros((arr.shape[0], arr.shape[1], 3), dtype=np.float32)
    for cls, col in PALETTE.items(): rgb[arr == cls] = col
    return rgb

def save_outputs(land: np.ndarray, base_out_dir: Path, fdict: dict):
    orig_snow   = int(round(fdict['snow']   * 100, 0))
    orig_algae  = int(round(fdict['algae']  * 100, 0))
    orig_moss   = int(round(fdict['moss']   * 100, 0))
    orig_lichen = int(round(fdict['lichen'] * 100, 0))
    tag = f"Snow_{orig_snow}_Algae_{orig_algae}_Moss_{orig_moss}_Lichen_{orig_lichen}"

    out_dir = base_out_dir / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    # array + csv
    np.save(out_dir / "lc_map.npy", land)
    pd.DataFrame(land).to_csv(out_dir / "lc_map.csv", header=False, index=False)

    # metadata
    meta = dict(size_m=int(land.shape[0]), kernel=KERNEL, seed=SEED, **fdict, classes=CLASS_NAMES)
    (out_dir / "metadata.json").write_text(json.dumps(meta, indent=2))

    # png preview with legend
    rgb = _colorize(land)
    fig, ax = plt.subplots(figsize=(6,6), dpi=200)
    ax.imshow(rgb, interpolation="nearest"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("Synthetic Land-Cover (1 m)")
    legend_handles = [Patch(color=PALETTE[c], label=f"{c} = {CLASS_NAMES[c]}") for c in sorted(PALETTE)]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=7, frameon=True)
    fig.tight_layout(); fig.savefig(out_dir / "lc_map.png", bbox_inches="tight"); plt.close(fig)
    return out_dir

# ---------- run ----------
BASE_OUT.mkdir(parents=True, exist_ok=True)
land, used_fracs = generate_landcover(
    size_m=SIZE_M, kernel=KERNEL,
    snow_frac=FRAC["snow"], algae_frac=FRAC["algae"],
    moss_frac=FRAC["moss"], lichen_frac=FRAC["lichen"],
    seed=SEED
)
out_dir = save_outputs(land, BASE_OUT, used_fracs)

# quick summary in notebook output
uniq, cnt = np.unique(land, return_counts=True)
print("Saved to:", out_dir)
print("Pixel counts:", dict(zip(uniq.tolist(), cnt.tolist())))

Saved to: G:\Hangkai\Anttarctic Vegetation Dynamic\DART\Antarctic_NDVI_all\Antarctic_NDVI_all\Data\LandCover\TableB\Snow_5_Algae_10_Moss_10_Lichen_10
Pixel counts: {0: 12500, 1: 162500, 2: 25000, 3: 25000, 4: 25000}
